# Analysis of ZeroSumNormal Constraint on Fourier Seasonality

This notebook demonstrates a problem with using `ZeroSumNormal` to constrain Fourier coefficients in the `StateSpaceTimeSeries` model. We show:

1. **Mathematical analysis**: Certain valid seasonal patterns cannot be fitted
2. **Residual structure**: The fitting residual is always a multiple of a specific function $g(t)$
3. **CausalPy impact**: Biased treatment effect estimates in Interrupted Time Series analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## Part 1: Mathematical Foundation

### The Fourier Basis

For period $S=12$ (monthly data with annual seasonality), `FrequencySeasonality` uses:
- Harmonics $j=1,2,\ldots,5$: both $\cos$ and $\sin$ (10 parameters)
- Harmonic $j=6$ (Nyquist): only $\cos$ (1 parameter)

Total: **11 parameters**

In [ ]:
S = 12  # Period (monthly)
n_params = 11

def build_fourier_basis(t, S=12):
    """Build Fourier basis matching pymc-extras FrequencySeasonality."""
    n = S // 2
    basis = []
    for j in range(1, n):  # j = 1 to 5
        basis.append(np.cos(2 * np.pi * j * t / S))
        basis.append(np.sin(2 * np.pi * j * t / S))
    basis.append(np.cos(2 * np.pi * n * t / S))  # Nyquist cos only
    return np.column_stack(basis)

t = np.arange(S)
X = build_fourier_basis(t, S)
print(f"Basis matrix shape: {X.shape}")

### The Unrepresentable Signal $g(t)$

The `ZeroSumNormal` constraint enforces $\sum_i \theta_i = 0$. Any signal component proportional to $g(t)$ (defined below) **cannot be represented**.

**Closed-form definition:**

$$g(t) = \begin{cases} 
n = S/2 & \text{if } t = 0 \\
0 & \text{if } t \text{ even}, t \neq 0 \\
\cot\left(\frac{\pi t}{S}\right) - 1 & \text{if } t \text{ odd}
\end{cases}$$

In [ ]:
def g_unrepresentable(t, S=12):
    """
    Closed-form for the unrepresentable signal.
    This is the function orthogonal to all zero-sum constrained signals.
    """
    t = np.asarray(t) % S
    result = np.zeros_like(t, dtype=float)
    result[t == 0] = S // 2
    odd_mask = (t % 2 == 1)
    result[odd_mask] = 1.0 / np.tan(np.pi * t[odd_mask] / S) - 1.0
    return result

g = g_unrepresentable(t, S)

# Display values
print("The unrepresentable signal g(t):")
print("-" * 50)
for ti, gi in zip(t, g):
    if ti == 0:
        formula = f"n = {S//2}"
    elif ti % 2 == 0:
        formula = "0"
    else:
        formula = f"cot(π·{ti}/{S}) - 1"
    print(f"  g({ti:2d}) = {gi:8.4f}  [{formula}]")
print("-" * 50)
print(f"  Mean: {np.mean(g):.6f} (zero-mean!)")
print(f"  Var:  {np.var(g):.4f}")

In [ ]:
# Visualize g(t)
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['crimson' if gi != 0 else 'lightgray' for gi in g]
ax.bar(t, g, color=colors, edgecolor='black', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Month (t)', fontsize=12)
ax.set_ylabel('g(t)', fontsize=12)
ax.set_title('The Unrepresentable Signal g(t) — Cannot Be Fitted by ZeroSumNormal Model', fontsize=14)
ax.set_xticks(t)
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.tight_layout()
plt.show()

## Part 2: Residual Structure

### Key Theorem

For **any** seasonal signal $y(t) = X\theta$, the residual from zero-sum constrained fitting is:

$$\boxed{\text{residual} = \overline{\theta} \cdot g(t)}$$

where $\overline{\theta} = \frac{1}{11}\sum_i \theta_i$ is the mean of the Fourier coefficients.

In [ ]:
def fit_with_constraint(y, X):
    """Fit y = X @ theta with zero-sum constraint."""
    theta_ols, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    theta_zs = theta_ols - np.mean(theta_ols)  # Project to zero-sum
    y_fit = X @ theta_zs
    return theta_zs, y_fit, theta_ols

### Example: Simple Sinusoidal Pattern

Consider $y(t) = 2\cos(\omega t) + 0.5\sin(2\omega t)$ where $\omega = 2\pi/12$.

In [ ]:
# Simple sinusoidal pattern
y_simple = 2 * np.cos(2 * np.pi * t / S) + 0.5 * np.sin(2 * np.pi * 2 * t / S)

theta_zs, y_fit_zs, theta_ols = fit_with_constraint(y_simple, X)
residual = y_simple - y_fit_zs

# The true coefficients
print("True Fourier coefficients θ:")
print(f"  θ = [2, 0, 0, 0.5, 0, 0, 0, 0, 0, 0, 0]")
print(f"  sum(θ) = {np.sum(theta_ols):.1f}")
print(f"  mean(θ) = {np.mean(theta_ols):.6f}")

# Verify residual = mean(θ) * g
mean_theta = np.mean(theta_ols)
predicted_residual = mean_theta * g

print(f"\nResidual verification:")
print(f"  Predicted: residual = mean(θ) · g = {mean_theta:.6f} · g")
print(f"  Actual residual matches predicted: {np.allclose(residual, predicted_residual)}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original signal
ax = axes[0]
ax.plot(t, y_simple, 'ko-', markersize=8, linewidth=2)
ax.set_title('Target: $y = 2\cos(\omega t) + 0.5\sin(2\omega t)$', fontsize=12)
ax.set_xlabel('Month')
ax.set_ylabel('Value')
ax.set_xticks(t)

# Zero-sum fit
ax = axes[1]
ax.plot(t, y_simple, 'ko-', markersize=8, linewidth=2, label='Target')
ax.plot(t, y_fit_zs, 'b^--', markersize=6, linewidth=2, label='Zero-sum fit')
ax.set_title(f'Fit (RMSE = {np.sqrt(np.mean(residual**2)):.4f})', fontsize=12)
ax.set_xlabel('Month')
ax.legend()
ax.set_xticks(t)

# Residual = mean(θ) * g
ax = axes[2]
ax.bar(t, residual, color='crimson', alpha=0.7, edgecolor='black', label='Residual')
ax.plot(t, predicted_residual, 'k--', linewidth=2, label=f'{mean_theta:.4f} · g(t)')
ax.set_title(f'Residual = mean(θ) · g(t) = {mean_theta:.4f} · g(t)', fontsize=12)
ax.set_xlabel('Month')
ax.legend()
ax.set_xticks(t)

plt.tight_layout()
plt.show()

## Part 3: Impact on CausalPy

### How StateSpaceTimeSeries Uses ZeroSumNormal

In `causalpy/pymc_models.py`, the `StateSpaceTimeSeries` model constrains the initial Fourier coefficients:

```python
_annual_seasonal = pm.ZeroSumNormal("params_freq", sigma=80, dims=annual_dims)
```

These coefficients become the initial state for the Kalman filter. The constraint means:

1. **Seasonal patterns with $\overline{\theta} \neq 0$ are systematically underfit**
2. **The missing component is always proportional to $g(t)$**
3. **This biases counterfactual predictions and treatment effect estimates**

### Demonstration: Biased Treatment Effect Estimation

We create synthetic data for an Interrupted Time Series with:
- A seasonal pattern that includes a $g(t)$ component
- A known treatment effect

We show that the zero-sum constraint leads to **biased treatment effect estimates**.

In [ ]:
# Create synthetic ITS data
np.random.seed(42)

# Parameters
n_pre = 36   # 3 years pre-intervention
n_post = 12  # 1 year post-intervention
n_total = n_pre + n_post

# Time index
dates = pd.date_range('2020-01-01', periods=n_total, freq='MS')
t_all = np.arange(n_total)
month = t_all % 12

# True seasonal pattern: includes g(t) component!
# θ_true has positive mean, so g(t) component exists
theta_true = np.array([3.0, 1.0, 2.0, 0.5, 1.5, 0.3, 1.0, 0.2, 0.8, 0.1, 0.5])
mean_theta_true = np.mean(theta_true)

print(f"True seasonal coefficients:")
print(f"  sum(θ) = {np.sum(theta_true):.2f}")
print(f"  mean(θ) = {mean_theta_true:.4f}")
print(f"  g(t) component amplitude = {mean_theta_true:.4f}")

# Build seasonal effect
X_full = build_fourier_basis(month, S)
seasonal_true = X_full @ theta_true

# Decompose into fittable + unfittable
theta_zs_true = theta_true - mean_theta_true
seasonal_fittable = X_full @ theta_zs_true
seasonal_unfittable = mean_theta_true * g_unrepresentable(month, S)

print(f"\nSeasonal variance decomposition:")
print(f"  Total variance: {np.var(seasonal_true):.4f}")
print(f"  Fittable variance: {np.var(seasonal_fittable):.4f}")
print(f"  Unfittable variance: {np.var(seasonal_unfittable):.4f}")
print(f"  % unfittable: {np.var(seasonal_unfittable)/np.var(seasonal_true)*100:.1f}%")

In [ ]:
# Add trend, treatment effect, and noise
trend = 100 + 0.1 * t_all  # Slight upward trend
treatment_effect_true = 5.0  # True causal effect
treatment = np.where(t_all >= n_pre, treatment_effect_true, 0)
noise = np.random.normal(0, 1.5, n_total)

# Full observed series
y_observed = trend + seasonal_true + treatment + noise

# Create DataFrame
df = pd.DataFrame({
    'y': y_observed,
    'trend': trend,
    'seasonal_true': seasonal_true,
    'seasonal_fittable': seasonal_fittable,
    'seasonal_unfittable': seasonal_unfittable,
    'treatment': treatment,
    'month': month
}, index=dates)

treatment_time = dates[n_pre]
print(f"Treatment time: {treatment_time}")
print(f"True treatment effect: {treatment_effect_true}")

In [ ]:
# Visualize the data
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Full series
ax = axes[0, 0]
ax.plot(dates, y_observed, 'ko-', markersize=4, linewidth=1)
ax.axvline(treatment_time, color='red', linestyle='--', linewidth=2, label='Treatment')
ax.set_title('Observed Time Series', fontsize=12)
ax.set_ylabel('y')
ax.legend()

# True seasonal pattern
ax = axes[0, 1]
ax.plot(range(12), seasonal_true[:12], 'ko-', markersize=8, linewidth=2, label='True seasonal')
ax.plot(range(12), seasonal_fittable[:12], 'b^--', markersize=6, linewidth=2, label='Fittable part')
ax.set_title('True Seasonal Pattern (One Cycle)', fontsize=12)
ax.set_xlabel('Month')
ax.set_ylabel('Effect')
ax.set_xticks(range(12))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.legend()

# Unfittable component
ax = axes[1, 0]
ax.bar(range(12), seasonal_unfittable[:12], color='crimson', alpha=0.7, edgecolor='black')
ax.set_title(f'Unfittable Component = {mean_theta_true:.2f} · g(t)', fontsize=12)
ax.set_xlabel('Month')
ax.set_ylabel('Effect')
ax.set_xticks(range(12))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])

# Impact on treatment effect
ax = axes[1, 1]
ax.text(0.5, 0.7, f'True treatment effect: {treatment_effect_true:.1f}', 
        transform=ax.transAxes, fontsize=14, ha='center')
ax.text(0.5, 0.5, f'Unfittable seasonal at treatment month:', 
        transform=ax.transAxes, fontsize=12, ha='center')
unfittable_at_treatment = seasonal_unfittable[n_pre % 12]
ax.text(0.5, 0.35, f'{mean_theta_true:.2f} · g({n_pre % 12}) = {unfittable_at_treatment:.2f}', 
        transform=ax.transAxes, fontsize=14, ha='center', color='crimson')
ax.text(0.5, 0.15, f'This bias propagates to treatment effect estimates!', 
        transform=ax.transAxes, fontsize=11, ha='center', style='italic')
ax.axis('off')
ax.set_title('Bias Mechanism', fontsize=12)

plt.tight_layout()
plt.show()

### Simulating the Zero-Sum Constraint Effect

We simulate what happens when the model can only fit the zero-sum component of seasonality.

**Key insight:** The function $g(t)$ has zero mean over a complete year. This means:
- Full-year average treatment effects may appear unbiased
- But **month-by-month estimates** and **partial-year analyses** show large biases
- The bias is largest in January (+6) and December (-4.7)

In [ ]:
# The bias equals mean(θ) * g(t) for each month
# Since g(t) has zero mean over 12 months, full-year averages look unbiased
# But partial-year and month-by-month analyses show large biases!

print("=" * 60)
print("MONTH-BY-MONTH BIAS IN TREATMENT EFFECT ESTIMATES")
print("=" * 60)
print(f"\\nTrue seasonal has mean(θ) = {mean_theta_true:.2f}")
print(f"Bias at month t = {mean_theta_true:.2f} × g(t)\\n")

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
g_vals = g_unrepresentable(np.arange(12), S)

print("Month-by-month biases:")
print("-" * 40)
for m in range(12):
    bias = mean_theta_true * g_vals[m]
    bar = "█" * int(abs(bias) * 2) if bias != 0 else ""
    sign = "+" if bias > 0 else "" if bias == 0 else ""
    print(f"  {month_names[m]:3s}: {sign}{bias:6.2f}  {bar}")
print("-" * 40)

print("\\n" + "=" * 60)
print("PARTIAL-YEAR ANALYSIS BIASES")
print("=" * 60)

# Different post-intervention periods
scenarios = [
    ("Jan-Jun (6 months)", list(range(0, 6))),
    ("Jul-Dec (6 months)", list(range(6, 12))),
    ("Jan-Mar (3 months)", list(range(0, 3))),
    ("Oct-Dec (3 months)", list(range(9, 12))),
    ("Full year (12 months)", list(range(12))),
]

print(f"\\nAverage bias for different post-intervention periods:")
print("-" * 50)
for name, months in scenarios:
    avg_bias = mean_theta_true * np.mean(g_unrepresentable(months, S))
    print(f"  {name:25s}: {avg_bias:+.2f}")
print("-" * 50)
print("\\n→ Partial-year analyses can have substantial bias!")

In [ ]:
# Demonstrate bias with PARTIAL-YEAR post-intervention (6 months: Jan-Jun)
# This is where the bias is most visible

n_post_partial = 6  # Only 6 months post-intervention
post_slice = slice(n_pre, n_pre + n_post_partial)

# True vs biased counterfactuals
counterfactual_true = trend[post_slice] + seasonal_true[post_slice] + noise[post_slice]
counterfactual_biased = trend[post_slice] + seasonal_fittable[post_slice] + noise[post_slice]

# Treatment effect estimates
observed_post = y_observed[post_slice]
effect_true = np.mean(observed_post - counterfactual_true)
effect_biased = np.mean(observed_post - counterfactual_biased)

# Expected bias from theory
expected_bias = mean_theta_true * np.mean(g_unrepresentable(month[post_slice], S))

print("Treatment Effect Estimates (6-month partial year: Jan-Jun)")
print("=" * 60)
print(f"  True treatment effect:       {treatment_effect_true:.2f}")
print(f"  Correct estimate (oracle):   {effect_true:.2f}")
print(f"  Biased estimate (zero-sum):  {effect_biased:.2f}")
print(f"  Actual bias:                 {effect_biased - treatment_effect_true:+.2f}")
print(f"  Theoretical bias:            {expected_bias:+.2f}")
print("=" * 60)
print(f"\n→ The {abs(effect_biased - treatment_effect_true):.0%} bias is substantial!")

In [ ]:
# Visualize the bias
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Counterfactual comparison (partial year)
ax = axes[0]
post_dates = dates[post_slice]
ax.plot(post_dates, observed_post, 'ko-', markersize=8, linewidth=1.5, label='Observed')
ax.plot(post_dates, counterfactual_true, 'g--', linewidth=2, label='True counterfactual')
ax.plot(post_dates, counterfactual_biased, 'r--', linewidth=2, label='Biased counterfactual')
ax.fill_between(post_dates, counterfactual_true, counterfactual_biased, 
                alpha=0.3, color='red', label='Bias region')
ax.set_title('Post-Intervention (Jan-Jun): Counterfactual Comparison', fontsize=12)
ax.set_ylabel('y')
ax.legend()

# Right: Treatment effect comparison
ax = axes[1]
x_pos = [0, 1, 2]
values = [treatment_effect_true, effect_true, effect_biased]
colors = ['green', 'blue', 'red']
labels = ['True\\nEffect', 'Oracle\\nEstimate', 'Zero-Sum\\nEstimate']
bars = ax.bar(x_pos, values, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Treatment Effect')
ax.set_title('Treatment Effect Estimation (6-month partial year)', fontsize=12)
ax.axhline(treatment_effect_true, color='green', linestyle='--', alpha=0.5)

# Add bias annotation
bias = effect_biased - treatment_effect_true
ax.annotate(f'Bias: {bias:+.2f}\\n({abs(bias/treatment_effect_true)*100:.0f}% error)', 
            xy=(2, effect_biased), xytext=(2.4, effect_biased),
            fontsize=12, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

## Part 4: The Constraint Has No Physical Meaning

The `ZeroSumNormal` constraint enforces:

$$\sum_{j=1}^{5} (a_j + b_j) + a_6 = 0$$

This has **no physical interpretation**. Compare with what would make sense:

| Constraint | Meaning | Already True? |
|------------|---------|---------------|
| $\sum_t \gamma(t) = 0$ | Zero-mean seasonal effect | **Yes** (automatic with $j \geq 1$) |
| $\sum_j (a_j + b_j) = 0$ | ??? | Arbitrary, harmful |

In [ ]:
# Verify: Fourier basis already has zero mean
print("Mean of each basis function (should all be ~0):")
for i in range(n_params):
    print(f"  Basis {i}: mean = {np.mean(X[:, i]):.10f}")
print("\n→ Seasonal effect automatically has zero mean. ZeroSumNormal is unnecessary!")

## Recommendation

Replace `ZeroSumNormal` with `Normal` in `StateSpaceTimeSeries`:

```python
# Current (problematic)
_annual_seasonal = pm.ZeroSumNormal("params_freq", sigma=80, dims=annual_dims)

# Recommended
_annual_seasonal = pm.Normal("params_freq", mu=0, sigma=80, dims=annual_dims)
```

This preserves the regularizing effect of the prior without imposing the arbitrary sum constraint.

## Summary

1. **The unrepresentable signal** $g(t)$ is defined by a simple closed form
2. **The residual** from any zero-sum fit equals $\overline{\theta} \cdot g(t)$
3. **CausalPy impact**: Treatment effect estimates are biased when the true seasonal pattern has $\overline{\theta} \neq 0$
4. **The constraint has no physical meaning** and should be replaced with a simple `Normal` prior